# Afternoon class 26/08 — Worksheet 10 SOLUTIONS: built-in functions, more practice   (L04)

Every cell below was executed on the same Python the lab ships; the quoted
output is real.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 10 — Built-in functions, more practice. Run this once.
cities = ["Toronto", "Ottawa", "Montreal", "Halifax"]
populations = [2794356, 1017449, 1762949]
scores = [88, 42, 95, 67, 73]
people = [("Ada", 36), ("Bo", 24), ("Cai", 41), ("Dee", 29)]
raw_ages = ["36", "24", "41"]

print("cities:     ", cities)
print("populations:", populations)
print("scores:     ", scores)
print("people:     ", people)

PART A — Measuring and converting

### Question 1

Warm-up. -> `4`, `365`, `95 42`, `73.0`, `<class 'float'>`.

Every one of these takes the whole collection; none of them needs a loop.

The average printed as `73.0` rather than `73` because `/` always produces
a float, even when the division comes out exact. That trailing `.0` is a
type showing through, not a rounding artefact.

In [ ]:
print(len(cities))
print(sum(scores))
print(max(scores), min(scores))
print(round(sum(scores) / len(scores), 2))
print(type(sum(scores) / len(scores)))   # / always gives a float

### Question 2

Converting before adding. -> `[36, 24, 41]`, `101`, `33.67`, then `3624`.

`"36" + "24"` is `"3624"`. Not an error, not a warning — string
concatenation, producing something that looks like a number and happens to
be four digits long.

This is the most common bug in code that reads from a file or a form,
because everything arrives as text and the mistake only surfaces when
somebody eventually reads the total. Convert at the boundary, the moment
the data arrives, not at the point of use.

In [ ]:
ages = [int(a) for a in raw_ages]
print(ages)
print(sum(ages))
print(round(sum(ages) / len(ages), 2))

print(raw_ages[0] + raw_ages[1])
# "36" + "24" is "3624" -- concatenation, not addition. No error is raised,
# which is exactly why this one survives into production.

PART B — Ordering, and choosing what to order by

### Question 3

Sorting by a key. -> `['Halifax', 'Montreal', 'Ottawa', 'Toronto']`, then `['Ottawa', 'Toronto', 'Halifax', 'Montreal']`, then `['Montreal', 'Toronto', 'Halifax', 'Ottawa']`, then `cities` unchanged.

`key=len` compares the LENGTH of each name while still returning the names
themselves — the key decides the comparison, never the output.

Look at the by-length result: `Toronto` and `Halifax` are both 7
characters, and they come back in their original relative order. Python's
sort is STABLE, which means ties keep their existing arrangement — that is
what makes it safe to sort by one field and then another.

In [ ]:
print(sorted(cities))
print(sorted(cities, key=len))
print(sorted(cities, key=len, reverse=True))
print(cities)   # sorted() always leaves the original alone

### Question 4

Sorting records. -> by name it is unchanged, then `[('Bo', 24), ('Dee', 29), ('Ada', 36), ('Cai', 41)]`, then that reversed.

With no `key`, `sorted(people)` compares the TUPLES themselves, which means
field 0 first — the name. That is why the first result looks unsorted and
is not: the input already happened to be in name order.

`key=lambda person: person[1]` switches the comparison to the age. A lambda
here is nothing more than "given one item, hand back the thing to compare".

In [ ]:
print(sorted(people))                                # by name -- field 0
print(sorted(people, key=lambda person: person[1]))  # by age
print(sorted(people, key=lambda person: person[1], reverse=True))

### Question 5

max and min with a key. -> `Montreal`, `Ottawa`, `('Cai', 41)`, `Cai`.

`max` returns the WHOLE item, not the value the key produced — which is why
you get the tuple `('Cai', 41)` and have to index it for the name. If you
only wanted the highest age, that is `max([p[1] for p in people])`.

Same `key=` idea as `sorted`, and it also works on `min`. One concept,
three functions — worth noticing so you learn it once.

In [ ]:
print(max(cities, key=len))
print(min(cities, key=len))

oldest = max(people, key=lambda person: person[1])
print(oldest)        # the whole tuple
print(oldest[0])     # just the name

PART C — Lazy iterators, combined

### Question 6

range and zip. -> `[0, 10, 20, 30, 40, 50]`, `[5, 4, 3, 2, 1]`, three pairs, then `3 4 3`.

FOUR CITIES, THREE POPULATIONS, THREE PAIRS. Halifax was dropped without a
word.

`zip` stops at the shortest input, and here that silently discarded the one
city with missing data — which is precisely the row you would most want
flagged. The `3 4 3` line is the only reason you can see it happened.

Compare the lengths before you zip anything you did not build yourself.

In [ ]:
print(list(range(0, 51, 10)))
print(list(range(5, 0, -1)))

pairs = list(zip(cities, populations))
print(pairs)
print(len(pairs), len(cities), len(populations))
# 3 pairs, not 4 -- Halifax was silently dropped when populations ran out.

### Question 7

dict(zip(...)). -> `{'Toronto': 2794356, 'Ottawa': 1017449, 'Montreal': 1762949}`, then `1017449`, then `False`.

`dict()` accepts any sequence of `(key, value)` pairs, and `zip` produces
exactly that — so the two compose into a one-line "build a lookup from two
parallel lists".

And `"Halifax" in city_pop` is `False`. Q6's silent drop has now been baked
into a dictionary that looks perfectly complete: three cities, three
populations, internally consistent. The data loss happened one line earlier
and left no trace at all here.

In [ ]:
city_pop = dict(zip(cities, populations))
print(city_pop)

print(city_pop["Ottawa"])
print("Halifax" in city_pop)   # False -- zip dropped it, so it never arrived

### Question 8

enumerate over sorted records. -> `1. Cai (41)`, `2. Ada (36)`, `3. Dee (29)`, `4. Bo (24)`, then that list reversed.

`for rank, (name, age) in enumerate(...)` unpacks two levels in a single
statement, because `enumerate` hands over `(rank, (name, age))`. The inner
brackets have to mirror the shape of the data — worksheet 13 Q6b makes the
same point where it is easier to get wrong.

`reversed()` returns a lazy object, so it needs `list()` to print usefully —
and reversing a descending sort gives you the ascending one back.

In [ ]:
by_age = sorted(people, key=lambda person: person[1], reverse=True)

for rank, (name, age) in enumerate(by_age, start=1):
    print(f"{rank}. {name} ({age})")

print(list(reversed(by_age)))

### Question 9

Stretch — map/filter, comprehensions, and the one-shot trap. -> `['88%', '42%', '95%', '67%', '73%']` and `[88, 95, 67, 73]`, each printed twice; then the three pairs, then `[]`.

Both spellings agree, as they must. The comprehension is what most Python
code uses, because the condition reads left to right without a `lambda`
standing in the way.

AND THE LAST LINE IS EMPTY. `report` was consumed by the first `list()`,
and a zip does not rewind. Nothing failed and nothing warned — you simply
get an empty list from a variable that visibly had data in it one line
earlier. Store `list(zip(...))` once if you need it twice.

In [ ]:
print(list(map(lambda s: f"{s}%", scores)))
print(list(filter(lambda s: s >= 60, scores)))

print([f"{s}%" for s in scores])
print([s for s in scores if s >= 60])

report = zip(cities, populations)
print(list(report))
print(list(report))   # empty -- the first list() consumed it